# Modality-Adaptive Attention for Vision-Language Models

This notebook demonstrates the key findings from our research on attention mechanisms for vision-text transformers.

## Key Insight

**Not all tokens need the same attention mechanism.** We propose routing attention based on modality:
- **Vision-to-Vision**: Linear attention (O(n) complexity) - works well for spatial relationships
- **Text-to-Vision**: Softmax attention - essential for precise grounding
- **Text-to-Text**: Softmax attention - required for language modeling

## Results Summary

| Variant | BLEU-1 | Perplexity | Status |
|---------|--------|------------|--------|
| Baseline (Softmax) | 0.220 | 314.95 | Best caption quality |
| Proposed (Modality-Adaptive) | 0.183 | 293.89 | **Lowest perplexity** |
| Ablation (Linear) | ~0 | 1.1 | **Collapsed** |

In [ ]:
# Setup
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import GPT2Tokenizer

# Project imports
from src.models.vision_text_model import create_model
from src.utils.checkpointing import load_checkpoint
from src.data.transforms import get_image_transform
from src.evaluation.visualization import visualize_attention, compare_attention_patterns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Models

We'll load two models for comparison:
- **Baseline**: Standard softmax attention everywhere
- **Proposed**: Modality-adaptive attention (linear for vision, softmax for text)

In [ ]:
def load_model(name, attention_type, checkpoint_path):
    """Load a model variant."""
    config = {
        "model": {
            "hidden_dim": 768,
            "num_heads": 12,
            "num_layers": 6,
            "attention_type": attention_type,
        }
    }
    
    model = create_model(config)
    load_checkpoint(checkpoint_path, model=model, device=device)
    model.eval()
    model.to(device)
    
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Loaded {name}: {num_params:,} trainable parameters")
    
    return model

# Load models
baseline_model = load_model(
    "Baseline (Softmax)", 
    "softmax", 
    project_root / "checkpoints/baseline_uniform_softmax/checkpoint-28000"
)

proposed_model = load_model(
    "Proposed (Modality-Adaptive)", 
    "modality_adaptive", 
    project_root / "checkpoints/proposed_modality_adaptive/checkpoint-28000"
)

## 2. Load a Sample Image

In [ ]:
# Load sample image
import matplotlib
sample_data_dir = Path(matplotlib.__file__).parent / "mpl-data" / "sample_data"
image_path = sample_data_dir / "grace_hopper.jpg"

image = Image.open(image_path).convert("RGB")
transform = get_image_transform(is_train=False)
image_tensor = transform(image).unsqueeze(0).to(device)

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title("Sample Image: Grace Hopper")
plt.axis('off')
plt.show()

## 3. Generate Captions

Let's compare captions generated by each model.

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

def generate_caption(model, image_tensor, max_length=30):
    """Generate a caption for an image."""
    with torch.no_grad():
        generated_ids = model.generate(
            images=image_tensor,
            max_length=max_length,
            do_sample=False,
        )
    caption = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return caption.strip()

# Generate captions
baseline_caption = generate_caption(baseline_model, image_tensor)
proposed_caption = generate_caption(proposed_model, image_tensor)

print("=" * 60)
print("Caption Comparison")
print("=" * 60)
print(f"\nBaseline (Softmax):")
print(f"  {baseline_caption}")
print(f"\nProposed (Modality-Adaptive):")
print(f"  {proposed_caption}")

## 4. Visualize Attention Patterns

Let's examine how attention patterns differ between the models.

In [ ]:
from src.data.collator import create_causal_mask

def get_attention_weights(model, image_tensor, num_tokens=5, layer_idx=5):
    """Extract attention weights during generation."""
    with torch.no_grad():
        vision_features = model.vision_encoder(image_tensor)
        vision_embeds = model.vision_projection(vision_features)
        
        generated_ids = torch.tensor([[tokenizer.eos_token_id]], device=device)
        all_attentions = []
        
        for _ in range(num_tokens):
            text_embeds = model.text_embeddings(generated_ids)
            hidden_states = torch.cat([vision_embeds, text_embeds], dim=1)
            seq_len = hidden_states.size(1)
            
            position_ids = torch.arange(seq_len, device=device)
            hidden_states = hidden_states + model.position_embeddings(position_ids)
            
            modality_mask = torch.zeros(1, seq_len, dtype=torch.long, device=device)
            modality_mask[:, model.num_vision_tokens:] = 1
            hidden_states = hidden_states + model.modality_embeddings(modality_mask)
            
            causal_mask = create_causal_mask(seq_len, model.num_vision_tokens, device)
            
            hidden_states, _, attentions = model.transformer(
                hidden_states,
                modality_mask=modality_mask,
                causal_mask=causal_mask,
                num_vision_tokens=model.num_vision_tokens,
                output_attentions=True,
            )
            
            if attentions and len(attentions) > layer_idx:
                all_attentions.append(attentions[layer_idx])
            
            logits = model.lm_head(hidden_states[:, -1:, :])
            next_token = logits.argmax(dim=-1)
            generated_ids = torch.cat([generated_ids, next_token], dim=1)
    
    return all_attentions[-1] if all_attentions else None

# Get attention weights
baseline_attn = get_attention_weights(baseline_model, image_tensor)
proposed_attn = get_attention_weights(proposed_model, image_tensor)

print("Attention weights extracted successfully!")

In [ ]:
# Visualize attention patterns
import seaborn as sns

def visualize_attention_comparison(baseline_attn, proposed_attn, num_vision_tokens=197, head_idx=0):
    """Compare attention patterns between baseline and proposed."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    models = [
        ("Baseline (Softmax)", baseline_attn),
        ("Proposed (Modality-Adaptive)", proposed_attn)
    ]
    
    for row, (name, attn) in enumerate(models):
        if attn is None:
            continue
            
        if attn.dim() == 4:
            attn = attn[0]
        
        attn = attn[head_idx].cpu().numpy()
        seq_len = attn.shape[0]
        
        # Vision-to-Vision
        v2v = attn[:num_vision_tokens, :num_vision_tokens]
        sns.heatmap(v2v, ax=axes[row, 0], cmap='viridis', cbar=True)
        axes[row, 0].set_title(f"{name}\nVision → Vision")
        
        # Text-to-Vision
        t2v = attn[num_vision_tokens:, :num_vision_tokens]
        sns.heatmap(t2v, ax=axes[row, 1], cmap='viridis', cbar=True)
        axes[row, 1].set_title("Text → Vision")
        
        # Text-to-Text
        t2t = attn[num_vision_tokens:, num_vision_tokens:]
        mask = np.triu(np.ones_like(t2t), k=1)
        sns.heatmap(t2t, ax=axes[row, 2], cmap='viridis', cbar=True, mask=mask)
        axes[row, 2].set_title("Text → Text (Causal)")
    
    plt.suptitle(f"Attention Pattern Comparison (Head {head_idx}, Layer 5)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_attention_comparison(baseline_attn, proposed_attn)

## 5. Attention Statistics

Let's analyze the entropy (distribution) of attention weights.

In [ ]:
def compute_attention_entropy(attn, num_vision_tokens=197):
    """Compute attention entropy for different regions."""
    if attn.dim() == 4:
        attn = attn[0]
    
    attn = attn.cpu().numpy()
    
    def entropy(p):
        p = np.clip(p, 1e-10, 1)
        return -np.sum(p * np.log(p), axis=-1)
    
    v2v = attn[:, :num_vision_tokens, :num_vision_tokens]
    t2v = attn[:, num_vision_tokens:, :num_vision_tokens]
    t2t = attn[:, num_vision_tokens:, num_vision_tokens:]
    
    return {
        'v2v': entropy(v2v).mean(),
        't2v': entropy(t2v).mean() if t2v.size > 0 else 0,
        't2t': entropy(t2t).mean() if t2t.size > 0 else 0,
    }

baseline_stats = compute_attention_entropy(baseline_attn)
proposed_stats = compute_attention_entropy(proposed_attn)

# Plot comparison
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(3)
width = 0.35

baseline_vals = [baseline_stats['v2v'], baseline_stats['t2v'], baseline_stats['t2t']]
proposed_vals = [proposed_stats['v2v'], proposed_stats['t2v'], proposed_stats['t2t']]

ax.bar(x - width/2, baseline_vals, width, label='Baseline (Softmax)', color='#2ecc71')
ax.bar(x + width/2, proposed_vals, width, label='Proposed (Modality-Adaptive)', color='#3498db')

ax.set_ylabel('Entropy')
ax.set_title('Attention Entropy by Region\n(Higher = More Distributed Attention)')
ax.set_xticks(x)
ax.set_xticklabels(['Vision→Vision', 'Text→Vision', 'Text→Text'])
ax.legend()

plt.tight_layout()
plt.show()

print("\nAttention Entropy Summary:")
print(f"{'Region':<15} {'Baseline':<12} {'Proposed':<12}")
print("=" * 40)
print(f"{'V→V':<15} {baseline_stats['v2v']:<12.4f} {proposed_stats['v2v']:<12.4f}")
print(f"{'T→V':<15} {baseline_stats['t2v']:<12.4f} {proposed_stats['t2v']:<12.4f}")
print(f"{'T→T':<15} {baseline_stats['t2t']:<12.4f} {proposed_stats['t2t']:<12.4f}")

## 6. Load and Display Results

Let's load the full experimental results.

In [ ]:
import json

results_dir = project_root / "results"

results = {}
for variant in ['baseline', 'proposed', 'ablation', 'tuned_v2', 'progressive']:
    result_files = list((results_dir / variant).glob('*_results.json'))
    if result_files:
        with open(result_files[0]) as f:
            results[variant] = json.load(f)

# Display results table
print("\n" + "=" * 80)
print("EXPERIMENTAL RESULTS SUMMARY")
print("=" * 80)
print(f"\n{'Variant':<30} {'BLEU-1':<10} {'ROUGE-L':<10} {'CIDEr':<10} {'Perplexity':<12}")
print("-" * 80)

display_names = {
    'baseline': 'Baseline (Softmax)',
    'proposed': 'Proposed (Modality-Adaptive)',
    'ablation': 'Ablation (Linear)',
    'tuned_v2': 'Tuned V2 (Gated Mixing)',
    'progressive': 'Progressive',
}

for variant, data in results.items():
    metrics = data['metrics']
    ppl = data['perplexity']
    name = display_names.get(variant, variant)
    
    bleu = metrics['BLEU-1']
    if bleu < 0.001:
        bleu_str = "~0"
    else:
        bleu_str = f"{bleu:.4f}"
    
    print(f"{name:<30} {bleu_str:<10} {metrics['ROUGE-L']:<10.4f} {metrics['CIDEr']:<10.4f} {ppl:<12.2f}")

## 7. Key Findings

### 1. Pure Linear Attention Collapses
The ablation model using uniform linear attention achieved near-zero BLEU scores and collapsed to a perplexity of 1.1, indicating complete model failure.

### 2. Modality-Adaptive Achieves Lowest Perplexity
Our proposed approach achieves the lowest perplexity (293.89), suggesting better language modeling despite slightly lower BLEU scores.

### 3. Trade-off Between Efficiency and Quality
The baseline achieves the highest BLEU-1 (0.220), but at the cost of O(n²) attention everywhere. The modality-adaptive approach offers a middle ground.

### Conclusion
**Not all tokens need the same attention mechanism.** Vision tokens benefit from efficient linear attention, while text tokens and cross-modal grounding require the precision of softmax attention.

In [ ]:
# Final visualization - bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

variants = ['baseline', 'proposed', 'ablation']
colors = ['#2ecc71', '#3498db', '#e74c3c']
labels = ['Baseline\n(Softmax)', 'Proposed\n(Modality-Adaptive)', 'Ablation\n(Linear)']

# BLEU-1
bleu_scores = [results[v]['metrics']['BLEU-1'] for v in variants]
axes[0].bar(labels, bleu_scores, color=colors)
axes[0].set_ylabel('BLEU-1 Score')
axes[0].set_title('Caption Quality (BLEU-1)')
axes[0].set_ylim(0, 0.25)

# Perplexity
perplexities = [results[v]['perplexity'] for v in variants]
axes[1].bar(labels, perplexities, color=colors)
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Language Model Quality (Perplexity)')

plt.suptitle('Model Comparison: Attention Mechanism Impact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNote: Lower perplexity is better. The ablation model collapsed (perplexity = 1.1).")